# Customer Segmentation & Behavior Analytics
### **Thiranex Internship — Project 2**

---

## 📌 1. Project Overview & Business Problem

Customer segmentation is the process of dividing a company's customers into distinct groups that reflect similarities among customers in each group. The primary goal of this project is to:
1. **Analyze customer demographic and purchase behavior** using unsupervised machine learning.
2. **Identify natural clusters** based on **Annual Income** and **Spending Score**.
3. **Profile customer personas** and formulate targeted, data-driven marketing strategies.

### Key Deliverables:
- Exploratory Data Analysis (EDA) on age, income, and spending patterns.
- Finding the optimal number of clusters using the **Elbow Method (WCSS)** and **Silhouette Score**.
- Model training using **K-Means Clustering** and **Hierarchical Clustering (Agglomerative)**.
- Actionable business recommendations and segment persona profiling.

## 🛠️ 2. Environment Setup & Library Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage

# Visualization settings
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 150

## 📊 3. Data Ingestion & Preprocessing

In [2]:
# Load dataset
df = pd.read_csv('../data/Mall_Customers.csv')
df.columns = ['CustomerID', 'Gender', 'Age', 'Annual_Income_k$', 'Spending_Score']

print("Shape of dataset:", df.shape)
df.head()

In [3]:
# Check data types and null values
df.info()
print("\nMissing values count:")
print(df.isnull().sum())

In [4]:
# Statistical summary
df.describe().T

## 📈 4. Exploratory Data Analysis (EDA)

Let us explore how demographic attributes relate to customer spending behavior.

In [5]:
# Distribution of Age, Income, and Spending Score
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(df['Age'], kde=True, ax=axes[0], color='#4361ee', bins=15)
axes[0].set_title('Age Distribution')

sns.histplot(df['Annual_Income_k$'], kde=True, ax=axes[1], color='#2ec4b6', bins=15)
axes[1].set_title('Annual Income Distribution ($k)')

sns.histplot(df['Spending_Score'], kde=True, ax=axes[2], color='#e71d36', bins=15)
axes[2].set_title('Spending Score Distribution (1-100)')
plt.tight_layout()
plt.show()

In [6]:
# Gender Breakdown
plt.figure(figsize=(7, 4))
sns.countplot(x='Gender', data=df, palette=['#ff70a6', '#70d6ff'])
plt.title('Gender Distribution of Shoppers')
plt.show()

In [7]:
# Correlation Heatmap
plt.figure(figsize=(8, 5))
sns.heatmap(df[['Age', 'Annual_Income_k$', 'Spending_Score']].corr(), annot=True, cmap='mako', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

## 🎯 5. Determining the Optimal Number of Clusters (k)

We use two primary mathematical techniques:
1. **The Elbow Method (WCSS)**: Measures the compactness of the clusters.
2. **Silhouette Score**: Measures how similar an object is to its own cluster compared to other clusters ($s \in [-1, 1]$).

In [8]:
X = df[['Annual_Income_k$', 'Spending_Score']].values

# Calculate WCSS and Silhouette Scores
wcss = []
sil_scores = []
k_range = range(2, 11)

for k in range(1, 11):
    km = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    km.fit(X)
    wcss.append(km.inertia_)
    if k >= 2:
        sil_scores.append(silhouette_score(X, km.labels_))

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Elbow Plot
ax1.plot(range(1, 11), wcss, marker='o', color='#2563eb', linewidth=2.5)
ax1.axvline(x=5, color='red', linestyle='--', label='Elbow Point (k=5)')
ax1.set_title('Elbow Method for Optimal k', fontsize=13, weight='bold')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia (WCSS)')
ax1.legend()

# Silhouette Plot
ax2.plot(k_range, sil_scores, marker='s', color='#10b981', linewidth=2.5)
ax2.axvline(x=5, color='red', linestyle='--', label='Max Silhouette (k=5: 0.554)')
ax2.set_title('Silhouette Score Analysis', fontsize=13, weight='bold')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.legend()

plt.tight_layout()
plt.show()

> **Conclusion**: Both the Elbow curve inflection point and the peak Silhouette Score ($0.554$) confirm that **$k = 5$ is the optimal number of clusters**.

## 🤖 6. K-Means Model Training & Cluster Persona Mapping

In [9]:
# Fit Final K-Means Model
optimal_k = 5
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=15)
df['Cluster'] = kmeans.fit_predict(X)
centroids = kmeans.cluster_centers_

# Map Clusters to Personas
persona_dict = {
    0: "Spenders / Enthusiasts",
    1: "Budget / Conservative",
    2: "Mainstream / Moderate",
    3: "Champions / VIP",
    4: "Savers / High Potential"
}
df['Persona'] = df['Cluster'].map(persona_dict)

print("Cluster Counts:")
print(df['Persona'].value_counts())

In [10]:
# Visualize K-Means Clusters
plt.figure(figsize=(12, 7))
colors = ['#f43f5e', '#64748b', '#0ea5e9', '#10b981', '#f59e0b']

for c in range(optimal_k):
    cluster_data = df[df['Cluster'] == c]
    plt.scatter(
        cluster_data['Annual_Income_k$'],
        cluster_data['Spending_Score'],
        s=80,
        color=colors[c],
        label=f"Cluster {c}: {persona_dict[c]} (n={len(cluster_data)})",
        edgecolor='black',
        linewidth=0.5,
        alpha=0.85
    )

# Plot Centroids
plt.scatter(centroids[:, 0], centroids[:, 1], s=250, c='yellow', marker='*', edgecolor='black', linewidth=1.5, label='Centroids')
plt.title('Customer Segments via K-Means (Income vs Spending Score)', fontsize=14, weight='bold')
plt.xlabel('Annual Income (k$)', fontsize=12)
plt.ylabel('Spending Score (1-100)', fontsize=12)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 🌳 7. Hierarchical Agglomerative Clustering (Dendrogram)

In [11]:
# Dendrogram using Ward's minimum variance method
linked = linkage(X, method='ward')

plt.figure(figsize=(14, 6))
dendrogram(linked, truncate_mode='lastp', p=30, leaf_rotation=45., leaf_font_size=10., show_contracted=True)
plt.axhline(y=150, color='r', linestyle='--', label='5 Cluster Cut-off Threshold')
plt.title('Hierarchical Clustering Dendrogram', fontsize=14, weight='bold')
plt.xlabel('Customer Samples')
plt.ylabel('Euclidean Distance')
plt.legend()
plt.tight_layout()
plt.show()

## 📋 8. Cluster Profiling & Strategic Business Insights

In [12]:
# Aggregate summary metrics per cluster
summary = df.groupby(['Cluster', 'Persona']).agg(
    Customer_Count=('CustomerID', 'count'),
    Avg_Age=('Age', 'mean'),
    Avg_Income_k=('Annual_Income_k$', 'mean'),
    Avg_Spending_Score=('Spending_Score', 'mean')
).reset_index()

summary['Share_%'] = (summary['Customer_Count'] / len(df) * 100).round(1)
summary.round(1)

### 💡 Actionable Marketing Recommendations:

| Cluster | Persona | Key Traits | Marketing Strategy |
|---|---|---|---|
| **Cluster 3** | **Champions / VIP** | High Income ($86.5k), High Spend (82.1) | **Premium VIP Loyalty**: Concierge service, early access to luxury product launches, bespoke private events. |
| **Cluster 0** | **Spenders / Enthusiasts** | Low Income ($25.7k), High Spend (79.4) | **Affordable Trendy Campaigns**: Flash discounts, BNPL (Buy-Now-Pay-Later) installments, youth social media influencer campaigns. |
| **Cluster 2** | **Mainstream / Moderate** | Mid Income ($55.5k), Mid Spend (49.4) | **Brand Retention**: Mass-market family promotions, seasonal festive discounts, standard reward cards. |
| **Cluster 4** | **Savers / High Potential** | High Income ($88.3k), Low Spend (16.9) | **Value-Driven Upsell**: High-end durable investments, warranties, utility-focused marketing showcasing return on investment. |
| **Cluster 1** | **Budget / Conservative** | Low Income ($25.7k), Low Spend (20.2) | **Cost-Efficiency & Value Packs**: Essential discount coupons, clearances, bundled offers. |

---
**End of Project Notebook** | Thiranex Student Internship Submission